In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time
import pandas as pd
import os
import base64
import traceback

# =========================================================
# CONFIGURACIÓN
# =========================================================
AVISOS_FM = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\INGRESAR DE FM\AVISOS_FM.xlsx"
download_dir = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\INGRESAR DE FM\OUT-PUT ARCHIVO AVISOS"

df_AVISOS_FM = pd.read_excel(AVISOS_FM)
df_AVISOS_FM["ESTADO"] = df_AVISOS_FM.get("ESTADO", "").astype(str)

options = webdriver.ChromeOptions()
options.add_experimental_option("prefs", {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
})

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

URL_LOGIN = "https://portalgfe.osinergmin.gob.pe/ntcse/JPage?page=envioPIN"
URL_FORM  = "https://portalgfe.osinergmin.gob.pe/FMAYOR/JPage?page=fmayor.gui.solicitud"

# =========================================================
# LOGIN
# =========================================================
driver.get(URL_LOGIN)

campo_usuario = wait.until(EC.presence_of_element_located((
    By.XPATH, "/html/body/table[2]/tbody/tr[2]/td[2]/form/table/tbody/tr[3]/td[3]/input"
)))
campo_password = driver.find_element(
    By.XPATH, "/html/body/table[2]/tbody/tr[2]/td[2]/form/table/tbody/tr[4]/td[3]/input"
)
boton_login = driver.find_element(
    By.XPATH, "//*[@id='login']/tbody/tr[6]/td[2]/input"
)

campo_usuario.clear()
campo_usuario.send_keys("CALSER@EDN")
campo_password.clear()
campo_password.send_keys("EDN098")

ActionChains(driver).double_click(boton_login).perform()
wait = WebDriverWait(driver, 20)
wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
print("🚀 Acceso completo al sistema")

# =========================================================
# PROCESAMIENTO
# =========================================================
for index, row in df_AVISOS_FM.iterrows():
    try:
        print("\nProcesando registro...")
        driver.get(URL_FORM)
        time.sleep(8)

        # Fecha inicio
        campo_inicio = wait.until(EC.presence_of_element_located((
            By.XPATH, "/html/body/div/form/table/tbody/tr[3]/td/table/tbody/tr[1]/td[2]/input"
        )))
        campo_inicio.clear()
        campo_inicio.send_keys(
            pd.to_datetime(row["Fecha y Hora de Inicio de Interrupción"])
            .strftime("%d/%m/%Y %H:%M:%S")
        )

        # Descripción
        campo_descripcion = driver.find_element(
            By.XPATH, "/html/body/div/form/table/tbody/tr[3]/td/table/tbody/tr[2]/td[2]/textarea"
        )
        campo_descripcion.clear()
        campo_descripcion.send_keys(
            "Afectacion a instalaciones MT a consecuencia de terceros"
        )

        # Instalación
        campo_instalacion = driver.find_element(
            By.XPATH, "/html/body/div/form/table/tbody/tr[3]/td/table/tbody/tr[3]/td[2]/input"
        )
        campo_instalacion.clear()
        campo_instalacion.send_keys(str(row["Instalación Afectada"]))

        # Reposición
        campo_reposicion = driver.find_element(
            By.XPATH, "/html/body/div/form/table/tbody/tr[3]/td/table/tbody/tr[4]/td[2]/input"
        )
        campo_reposicion.clear()
        campo_reposicion.send_keys(
            pd.to_datetime(row["Fecha y Hora de Reposición"])
            .strftime("%d/%m/%Y %H:%M:%S")
        )

        # Lugares
        campo_lugares = driver.find_element(
            By.XPATH, "/html/body/div/form/table/tbody/tr[3]/td/table/tbody/tr[5]/td[2]/textarea"
        )
        campo_lugares.clear()
        campo_lugares.send_keys(str(row["Lugares y Sectores Afectados"]))

        # Selects
        Select(driver.find_element(
            By.XPATH, "/html/body/div/form/table/tbody/tr[3]/td/table/tbody/tr[6]/td[2]/select"
        )).select_by_index(1)

        Select(driver.find_element(
            By.XPATH, "/html/body/div/form/table/tbody/tr[3]/td/table/tbody/tr[7]/td[2]/select"
        )).select_by_index(3)

        # Ingresar
        boton_ingresar = driver.find_element(
            By.XPATH, "//*[@id='settings']/table/tbody/tr[4]/td/div/input"
        )
        ActionChains(driver).double_click(boton_ingresar).perform()
        time.sleep(6)

        # =========================================================
        # 🔥 PDF DEFINITIVO (OSINERGMIN = window.print)
        # =========================================================
        boton_pdf = wait.until(EC.element_to_be_clickable((
            By.XPATH, "//*[@id='codesiteContent']/table[2]/tbody/tr[3]/td/p/a"
        )))

        # Click (solo para consistencia visual)
        driver.execute_script("arguments[0].click();", boton_pdf)
        time.sleep(2)

        # Generar PDF real desde Chrome
        pdf = driver.execute_cdp_cmd("Page.printToPDF", {
            "printBackground": True
        })

        dia = pd.to_datetime(
            row["Fecha y Hora de Inicio de Interrupción"]
        ).strftime("%d")
        instalacion = str(row["Instalación Afectada"]).strip()
        nombre_pdf = f"D{dia} {instalacion}.pdf"
        ruta_pdf = os.path.join(download_dir, nombre_pdf)

        with open(ruta_pdf, "wb") as f:
            f.write(base64.b64decode(pdf["data"]))

        print(f"📄 PDF generado correctamente: {nombre_pdf}")

        df_AVISOS_FM.at[index, "ESTADO"] = "CORRECTO"
        print("✓ Registro ingresado correctamente")

    except Exception:
        traceback.print_exc()
        df_AVISOS_FM.at[index, "ESTADO"] = "ERROR"

# =========================================================
# GUARDAR RESULTADOS
# =========================================================
df_AVISOS_FM.to_csv("AVISOS_FM_procesado.csv", index=False)
print("\nArchivo guardado: AVISOS_FM_procesado.csv")
print("Proceso completado")

driver.quit()


🚀 Acceso completo al sistema

Procesando registro...
📄 PDF generado correctamente: D02 H-02.pdf
✓ Registro ingresado correctamente

Procesando registro...
📄 PDF generado correctamente: D02 CY-01.pdf
✓ Registro ingresado correctamente

Procesando registro...
📄 PDF generado correctamente: D03 SU-54.pdf
✓ Registro ingresado correctamente

Procesando registro...
📄 PDF generado correctamente: D04 AD-01.pdf
✓ Registro ingresado correctamente

Procesando registro...
📄 PDF generado correctamente: D04 CV-01.pdf
✓ Registro ingresado correctamente

Procesando registro...
📄 PDF generado correctamente: D03 CV-15.pdf
✓ Registro ingresado correctamente

Procesando registro...
📄 PDF generado correctamente: D03 W-02.pdf
✓ Registro ingresado correctamente

Archivo guardado: AVISOS_FM_procesado.csv
Proceso completado
